# Fase 4 — Análisis multivariado, contraste de hipótesis y baseline P6

| Campo | Valor |
|---|---|
| **Entrada** | `compounds_features.csv` (107 compuestos) + `activities_clean.csv` (baseline P6) |
| **Salidas** | `stats_tests.csv`, `clustering_summary.json`, `baseline_honest_metrics.csv`, figuras PCA/clustering/baseline |
| **Doc** | [`docs/fases/fase4_modelado.md`](../docs/fases/fase4_modelado.md) (§12 = baseline) |

## Por qué no hay modelado supervisado como producto principal

- `activity_class` es binarización de `pchembl_value >= 6` → **circular**.
- `pchembl_value` mezcla 13 endpoints → **no comparable** como target único.
- Split por filas = **fuga** (métricas infladas). Split por compuesto → **no generaliza**.

El **baseline predictivo honesto (P6)** se ejecuta en la **§4** de este mismo notebook.


## 0. Configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from src.paths import PROJECT_ROOT as ROOT, setup_path
setup_path()

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 120})

FIG_DIR = ROOT / "outputs" / "chembl" / "figures"
RESULTS_DIR = ROOT / "outputs" / "chembl" / "results"
for d in (FIG_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

import json
from scipy.cluster import hierarchy

from src.analisis_proyecto.chembl_baseline import (
    honest_baseline_compound_level,
    leaky_baseline_row_level,
)
from src.analisis_proyecto.chembl_multivariate import (
    FEATURE_COLS,
    cluster_vs_family_ari,
    kruskal_by_family,
    posthoc_dunn,
    run_kmeans_silhouette,
    run_pca,
    scale_features,
)
from src.analisis_proyecto.chembl_preprocessing import load_bioactivity

COMPOUNDS_CSV = ROOT / "data" / "processed" / "compounds_features.csv"
ACTIVITIES_CSV = ROOT / "data" / "processed" / "activities_clean.csv"
assert COMPOUNDS_CSV.exists(), "Ejecuta fase2_limpieza.ipynb primero"
assert ACTIVITIES_CSV.exists(), "Ejecuta fase2_limpieza.ipynb primero"

compounds = pd.read_csv(COMPOUNDS_CSV)
activities = load_bioactivity(ACTIVITIES_CSV)
print(f"Compuestos: {len(compounds)} | familias: {compounds['family'].nunique()}")


## 1. PCA (descriptores escalados)

In [ ]:
X = scale_features(compounds)
pca = run_pca(X, 2)
ev = pca["explained_variance_ratio"]
print(f"Varianza explicada PC1+PC2: {sum(ev)*100:.1f}%")
display(pd.DataFrame({
    "componente": ["PC1", "PC2"],
    "varianza": ev,
    "loadings": pca["loadings"],
}))

fig, ax = plt.subplots(figsize=(8, 6))
for fam in compounds["family"].unique():
    m = compounds["family"] == fam
    ax.scatter(pca["coords"][m, 0], pca["coords"][m, 1], label=fam, alpha=0.85)
ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")
ax.set_title("PCA — descriptores moleculares")
ax.legend(bbox_to_anchor=(1.05, 1), fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_scatter.png", bbox_inches="tight")
plt.show()


## 2. Clustering

In [ ]:
km = run_kmeans_silhouette(X)
print("Silhouette por k:", km["silhouette_by_k"])
print("best_k:", km["best_k"])

fig, ax = plt.subplots()
ax.plot(list(km["silhouette_by_k"].keys()), list(km["silhouette_by_k"].values()), "o-")
ax.set_xlabel("k")
ax.set_ylabel("silhouette")
ax.set_title("Selección de k (K-means)")
plt.tight_layout()
plt.savefig(FIG_DIR / "cluster_silhouette.png", bbox_inches="tight")
plt.show()

Z = hierarchy.linkage(X, method="ward")
fig, ax = plt.subplots(figsize=(12, 5))
hierarchy.dendrogram(Z, ax=ax, truncate_mode="lastp", p=20)
ax.set_title("Clustering jerárquico (Ward)")
plt.tight_layout()
plt.savefig(FIG_DIR / "dendrogram.png", bbox_inches="tight")
plt.show()

labels = km["labels"]
ari = cluster_vs_family_ari(labels, compounds["family"])
print(f"ARI clusters vs family: {ari:.3f}")
compounds["cluster"] = labels
compounds.to_csv(COMPOUNDS_CSV, index=False)


## 3. Tests Kruskal-Wallis + post-hoc Dunn

In [ ]:
test_vars = FEATURE_COLS + ["pchembl_median"]
stats_rows = []
for col in test_vars:
    res = kruskal_by_family(compounds, col)
    stats_rows.append(res)
    if res.get("p") is not None and res["p"] < 0.05:
        print(f"\nPost-hoc Dunn — {col} (p={res['p']:.4f}):")
        display(posthoc_dunn(compounds, col).round(4))

stats_df = pd.DataFrame(stats_rows)
display(stats_df.round(4))
stats_df.to_csv(RESULTS_DIR / "stats_tests.csv", index=False)

summary = {
    "best_k": km["best_k"],
    "silhouette_by_k": km["silhouette_by_k"],
    "ari_vs_family": ari,
}
(RESULTS_DIR / "clustering_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, col in enumerate(test_vars[:6]):
    sns.boxplot(data=compounds, x="family", y=col, ax=axes.flatten()[i])
    axes.flatten()[i].tick_params(axis="x", rotation=45)
    axes.flatten()[i].set_title(col)
plt.tight_layout()
plt.savefig(FIG_DIR / "family_boxplots_annotated.png", bbox_inches="tight")
plt.show()


## 4. Baseline predictivo honesto (P6)

> Demuestra que los descriptores clásicos **no generalizan** con split por compuesto → puente al proyecto GNN de la JIC.
> **NO** reportar el split por filas como resultado válido (fuga de datos).


In [ ]:
honest = honest_baseline_compound_level(compounds)
leaky = leaky_baseline_row_level(activities)

metrics = pd.DataFrame([honest, leaky])
display(metrics.round(4))
metrics.to_csv(RESULTS_DIR / "baseline_honest_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(metrics["split"], metrics["r2_test"], color=["#2ca02c", "#d62728"])
ax.set_ylabel("R² test")
ax.set_title("Baseline RF — split honesto vs fuga")
ax.axhline(0, color="gray", lw=0.8)
plt.tight_layout()
plt.savefig(FIG_DIR / "baseline_honest_vs_leaky.png", bbox_inches="tight")
plt.show()

### Conclusión P6

- **Split por compuesto (honesto):** R² bajo o negativo con 107 compuestos y 8 descriptores.
- **Split por filas (fuga):** R² artificialmente alto — la misma molécula aparece en train y test.
- Con esta señal limitada, los **grafos moleculares (GNN)** son la vía prometedora (proyecto JIC hermano).

---
*Anterior:* [`fase3_eda.ipynb`](fase3_eda.ipynb) · *Siguiente:* [`fase5_dashboard.ipynb`](fase5_dashboard.ipynb)